In [ ]:
# import essential
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [25]:
from general_utils import find_ephys_sessions
from pathlib import Path
from ephys_utils import collect_units_locations_to_csv

# Step 1: discover sessions
_, _, sessions = find_ephys_sessions()
print(f"Total sessions found: {len(sessions)}")


# Step 2: export unit locations
OUT_CSV = Path("/root/capsule/scratch/unit_locations_by_session.csv")

df_unitloc = collect_units_locations_to_csv(
    sessions,
    out_csv=OUT_CSV,
    verbose=True,
)

print("Done.")


Successfully read ephys NWB from: /root/capsule/data/ecephys_795396_2025-09-20_13-11-19_sorted_2025-10-29_20-43-11/nwb/ecephys_795396_2025-09-20_13-11-19_experiment1_recording1.nwb
Found behavior NWB: /root/capsule/data/optogenetics_nwb/795396_2025-09-20_13-11-19.nwb
Successfully read behavior NWB from: /root/capsule/data/optogenetics_nwb/795396_2025-09-20_13-11-19.nwb
Successfully appended units table to behavior NWB.


In [ ]:
sessions

In [ ]:
df_unitloc

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

from nwb_utils import NWBUtils
from ephys_utils import append_units_locations
from general_utils import extract_session_name_core


def _try_close(obj: Any) -> None:
    if obj is None:
        return
    try:
        close = getattr(obj, "close", None)
        if callable(close):
            close()
    except Exception:
        pass


def _extract_brain_region_from_ccf(loc: Any) -> str:
    # Try to extract a readable brain region string from ccf_location
    if loc is None:
        return ""
    if isinstance(loc, dict):
        br = loc.get("brain_region", "")
        return "" if br is None else str(br)
    try:
        return str(loc)
    except Exception:
        return ""


def _build_summary_lookup(
    summary: pd.DataFrame,
    *,
    summary_session_col: str,
    summary_cols: Optional[Sequence[str]],
    prefix: str,
) -> pd.DataFrame:
    """
    Build a per-session lookup table keyed by session_name_core.
    - If multiple rows map to the same core, keep the first.
    - Prefix all attached columns to avoid collisions.
    """
    if summary_session_col not in summary.columns:
        raise KeyError(f"summary is missing session column: {summary_session_col!r}")

    s = summary.copy()
    s[summary_session_col] = s[summary_session_col].astype(str)
    s["session_name_core"] = s[summary_session_col].map(
        lambda x: extract_session_name_core(x.rstrip("/").split("/")[-1])
    )

    if summary_cols is None:
        cols = [c for c in s.columns if c not in {summary_session_col, "session_name_core"}]
    else:
        cols = [c for c in summary_cols if c in s.columns]

    keep = ["session_name_core"] + cols
    s = s[keep].drop_duplicates(subset=["session_name_core"], keep="first")

    rename_map = {c: f"{prefix}{c}" for c in cols}
    s = s.rename(columns=rename_map)
    return s


def collect_units_locations_to_csv(
    sessions: Sequence[str],
    *,
    out_csv: str | Path,
    verbose: bool = True,
    # NEW: optional summary table to attach per session
    summary: Optional[pd.DataFrame] = None,
    summary_session_col: str = "session_name",
    summary_cols: Optional[Sequence[str]] = None,
    summary_prefix: str = "summary__",
) -> pd.DataFrame:
    """
    For each session in `sessions`:
      - load NWB
      - append_units_locations
      - output one row per unit with location fields

    NEW:
      - If `summary` is provided, attach per-session summary fields to each unit row
        (matched by extract_session_name_core).
    IMPORTANT:
      - session_name column == input session string from `sessions`.
    """
    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    summary_lookup: Optional[pd.DataFrame] = None
    summary_map: Optional[Dict[str, Dict[str, Any]]] = None

    if summary is not None:
        summary_lookup = _build_summary_lookup(
            summary,
            summary_session_col=summary_session_col,
            summary_cols=summary_cols,
            prefix=summary_prefix,
        )
        # Convert to dict for fast lookup in the loop
        summary_map = (
            summary_lookup.set_index("session_name_core")
            .to_dict(orient="index")
        )

    rows: List[Dict[str, Any]] = []
    failures: List[Tuple[str, str]] = []

    for i, session_input in enumerate(list(sessions), start=1):
        io = None
        nwb_data = None
        try:
            nwb_data, io = NWBUtils.combine_nwb(session_name=session_input)

            sess_raw = getattr(nwb_data, "session_id", None)
            sess_core = extract_session_name_core(
                sess_raw if sess_raw is not None else session_input
            )

            nwb_data = append_units_locations(nwb_data, sess_core)

            units = nwb_data.units
            n_units = len(units)

            # Prefer NWB unit ids if present; otherwise fall back to unit_index
            unit_ids: Optional[List[int]] = None
            try:
                if hasattr(units, "id"):
                    unit_ids = [int(x) for x in units.id[:]]
            except Exception:
                unit_ids = None

            has_ccf = "ccf_location" in units.colnames

            # Pull session-level summary fields once per session
            session_summary_fields: Dict[str, Any] = {}
            if summary_map is not None:
                session_summary_fields = summary_map.get(sess_core, {})

            for u in range(n_units):
                loc = None
                if has_ccf:
                    try:
                        loc = units["ccf_location"][u]
                    except Exception:
                        loc = None

                brain_region = _extract_brain_region_from_ccf(loc)

                row = {
                    "session_name": session_input,          # EXACT input string
                    "session_name_core": sess_core,         # useful for joining/debug
                    "unit_index": int(u),
                    "unit_id": int(unit_ids[u]) if unit_ids is not None and u < len(unit_ids) else int(u),
                    "ccf_location": loc,
                    "brain_region": brain_region,
                }

                # Attach summary fields (prefixed) if available
                if session_summary_fields:
                    row.update(session_summary_fields)

                rows.append(row)

            if verbose:
                print(f"[{i:>4}/{len(sessions)}] OK   {session_input}  n_units={n_units}")

        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            failures.append((str(session_input), msg))
            if verbose:
                print(f"[{i:>4}/{len(sessions)}] FAIL {session_input}  {msg}")

        finally:
            _try_close(io)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)

    if verbose:
        print(f"\nSaved: {out_csv}")
        if failures:
            print(f"[WARN] Failed sessions: {len(failures)} (showing up to 10)")
            for s, m in failures[:10]:
                print("  ", s, "->", m)

    return df


In [ ]:
from general_utils import find_ephys_sessions
from behavior_qc_visualization import load_behavior_model_summary_csv
from behavior_qc_metrics_summary import append_model_criteria_result

# Sessions
_, _, sessions = find_ephys_sessions()

# Summary
summary = load_behavior_model_summary_csv("/root/capsule/scratch/behavior_model_summary_ephys_sessions.csv")
summary = append_model_criteria_result(summary)

OUT_CSV = "/root/capsule/scratch/unit_locations_with_behavior_summary.csv"

df_unitloc = collect_units_locations_to_csv(
    sessions,  # test
    out_csv=OUT_CSV,
    summary=summary,
    summary_session_col="session_name",
    # summary_cols=None means "attach everything except session_name"
    summary_cols=[
        "QLearning_L1F1_CK1_softmax_pass_all_criteria",
        "best_model_name",
        "n_trials",
    ],
    summary_prefix="summary__",
    verbose=True,
)

df_unitloc.head()


In [ ]:
from __future__ import annotations

from typing import Optional, Sequence
import pandas as pd

from general_utils import extract_session_name_core


def append_summary_to_df_by_session(
    df: pd.DataFrame,
    summary: pd.DataFrame,
    *,
    df_session_col: str = "session_name",
    summary_session_col: str = "session_name",
    summary_cols: Optional[Sequence[str]] = None,
    prefix: str = "summary__",
    keep_session_core_cols: bool = False,
    how: str = "left",
    dedup_summary: bool = True,
) -> pd.DataFrame:
    """
    Append session-level summary columns to a DataFrame by matching sessions.

    Matching rule
    -------------
    - Both df and summary are matched using a normalized key:
        session_name_core = extract_session_name_core(basename(session_name))
      This is robust to:
        - full paths vs basenames
        - trailing slashes
        - extra prefixes/suffixes in session strings (as handled by extract_session_name_core)

    Parameters
    ----------
    df
        Input DataFrame that contains a session identifier column (df_session_col).

    summary
        Session-level summary DataFrame (e.g., behavior_model_summary_*.csv after
        append_model_criteria_result), containing summary_session_col.

    df_session_col
        Column in df containing session identifiers.

    summary_session_col
        Column in summary containing session identifiers.

    summary_cols
        Which columns from summary to append.
        - None: append all columns except summary_session_col.
        - Sequence: append only these columns (if present).

    prefix
        Prefix to add to appended summary columns to avoid name collisions.

    keep_session_core_cols
        If True, keep helper columns:
          - df_session_name_core
          - summary_session_name_core
        If False, drop helper columns before returning.

    how
        Merge mode. Usually "left".

    dedup_summary
        If True, drop duplicate sessions in summary (by core key) keeping first.

    Returns
    -------
    df_out
        A copy of df with appended summary columns.
    """
    if df_session_col not in df.columns:
        raise KeyError(f"df is missing required column: {df_session_col!r}")
    if summary_session_col not in summary.columns:
        raise KeyError(f"summary is missing required column: {summary_session_col!r}")

    df_out = df.copy()
    summ = summary.copy()

    # Normalize both session columns to session core keys
    df_out["_df_session_name_core"] = (
        df_out[df_session_col]
        .astype(str)
        .map(lambda s: extract_session_name_core(s.rstrip("/").split("/")[-1]))
    )

    summ["_summary_session_name_core"] = (
        summ[summary_session_col]
        .astype(str)
        .map(lambda s: extract_session_name_core(s.rstrip("/").split("/")[-1]))
    )

    # Decide which summary columns to append
    if summary_cols is None:
        cols_to_add = [c for c in summ.columns if c not in {summary_session_col, "_summary_session_name_core"}]
    else:
        cols_to_add = [c for c in summary_cols if c in summ.columns]

    # Build lookup table: one row per session core
    lookup = summ[["_summary_session_name_core"] + cols_to_add].copy()
    if dedup_summary:
        lookup = lookup.drop_duplicates(subset=["_summary_session_name_core"], keep="first")

    # Prefix columns to avoid collisions
    rename_map = {c: f"{prefix}{c}" for c in cols_to_add}
    lookup = lookup.rename(columns=rename_map)

    # Merge
    df_out = df_out.merge(
        lookup,
        left_on="_df_session_name_core",
        right_on="_summary_session_name_core",
        how=how,
    )

    # Optionally drop helper keys
    if not keep_session_core_cols:
        df_out = df_out.drop(columns=["_df_session_name_core", "_summary_session_name_core"], errors="ignore")

    return df_out


In [ ]:
summary['session_name']

In [ ]:
# df_unitloc could be output from collect_units_locations_to_csv (or any df with session_name)
df_with_summary = append_summary_to_df_by_session(
    df_unitloc,
    summary,
    df_session_col="session_name",
    summary_session_col="session_name",
    summary_cols=[
        "QLearning_L1F1_CK1_softmax_pass_all_criteria",
    ],
    prefix="summaryy__",
)

df_with_summary.head()
